In [ ]:
%md
## Notebook to run sparklens on a cluster in python
---
In order to run this, you need to:
- build the sparklens jar by issuing 'sbt assembly' at a command prompt. this will build a jar file in target/scala-2.11/sparklens-assembly-0.1.jar
- add this jar file to a Volume and install it to the cluster you want to run it on 
- install the org.json4s:json4s-jackson_2.13:4.1.0-M8 also to the same cluster

In [0]:
from pyspark.sql.functions import col, sum, avg
import time

QNL = sc._jvm.com.qubole.sparklens.QuboleNotebookListener.registerAndGet(sc._jsc.sc())

if (QNL.estimateSize() != QNL.getMaxDataSize()):
    QNL.purgeJobsAndStages()
    startTime = int(round(time.time() * 1000))

    df = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/catadb360dev/schemaadb360dev/bronze/flights-1m.csv")
    aggregated_df = df.groupBy("DISTANCE", "AIR_TIME").agg(
        avg("DISTANCE").alias("avg_distance"),
        avg("AIR_TIME").alias("avg_airtime")
    )

    aggregated_df.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable("avgdistandair") \

    endTime = int(round(time.time() * 1000))
    time.sleep(QNL.getWaiTimeInSeconds())
    print(QNL.getStats(startTime, endTime))